# 🛡️ Fraud Shield on IBM Z — Exploratory Data Analysis

> **IBM Datathon · Real-Time AI for Critical Decisions**

This notebook explores the ULB Credit Card Fraud dataset to understand
its structure, class imbalance, feature distributions, and temporal
patterns — all of which inform our model design.

## 1 · Setup & Imports

In [ ]:
import sys, os

# Let Python find the project root so we can import src.config
# (the notebook lives in notebooks/, project root is one level up)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Use our central config — no hardcoded paths
from src.config import DATA_PATH, TARGET_COL, PCA_FEATURES

# Plotting defaults
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
print("✅ Imports ready  |  Data path:", DATA_PATH)

## 2 · Load the Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"✅ Loaded {len(df):,} rows × {df.shape[1]} columns")

## 3 · Shape, Data Types & Missing Values

In [ ]:
# Quick overview of dimensions
print(f"Shape : {df.shape[0]:,} rows  ×  {df.shape[1]} columns\n")

# Data types — are there unexpected non-numeric columns?
print("Data types:")
print(df.dtypes.value_counts().to_string())

In [ ]:
# Check for missing values
missing = df.isnull().sum()
total_missing = missing.sum()

if total_missing == 0:
    print("✅ No missing values in any column.")
else:
    print(f"⚠️  {total_missing} missing values found:")
    print(missing[missing > 0])

**Insight:** The dataset has **284,807 transactions** across **31 numeric columns** (Time, V1–V28, Amount, Class) with **zero missing values**, so we can move straight to modelling without imputation.

## 4 · Class Balance — Fraud vs Legitimate

In [ ]:
# Counts and percentages
class_counts = df[TARGET_COL].value_counts()
class_pct    = df[TARGET_COL].value_counts(normalize=True) * 100

summary = pd.DataFrame({
    "Count":      class_counts,
    "Percentage": class_pct.round(4)
})
summary.index = ["Legitimate (0)", "Fraud (1)"]
print(summary.to_string())
print(f"\nFraud-to-Legit ratio:  1 : {class_counts[0] // class_counts[1]}")

In [ ]:
# Bar chart — class distribution
fig, ax = plt.subplots(figsize=(6, 4))

colors = ["#2ecc71", "#e74c3c"]
bars = ax.bar(["Legitimate (0)", "Fraud (1)"],
              class_counts.values, color=colors, edgecolor="black")

# Annotate each bar with count + percentage
for bar, count, pct in zip(bars, class_counts.values, class_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2000,
            f"{count:,}\n({pct:.3f}%)", ha="center", fontweight="bold", fontsize=10)

ax.set_title("Class Distribution — Extreme Imbalance", fontsize=14)
ax.set_ylabel("Number of Transactions")
fig.tight_layout()
plt.show()

**Insight:** Only **0.17 %** of transactions are fraudulent — a roughly **1 : 577** imbalance. Standard accuracy metrics would be misleading; we'll need techniques like **SMOTE** for resampling and metrics like **AUPRC** and **F1-score** to evaluate the model fairly.

## 5 · Transaction Amount — Fraud vs Non-Fraud

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# --- Left panel: full range (log-scale y for visibility) ---
for cls, color, label in [(0, "#2ecc71", "Legitimate"), (1, "#e74c3c", "Fraud")]:
    axes[0].hist(df[df[TARGET_COL] == cls]["Amount"],
                 bins=80, alpha=0.65, color=color, label=label)
axes[0].set_yscale("log")
axes[0].set_title("Amount Distribution — Full Range (log scale)")
axes[0].set_xlabel("Transaction Amount ($)")
axes[0].set_ylabel("Count (log)")
axes[0].legend()

# --- Right panel: zoom to < $500 where most fraud lives ---
for cls, color, label in [(0, "#2ecc71", "Legitimate"), (1, "#e74c3c", "Fraud")]:
    subset = df[(df[TARGET_COL] == cls) & (df["Amount"] < 500)]
    axes[1].hist(subset["Amount"], bins=60, alpha=0.65, color=color, label=label)
axes[1].set_title("Amount Distribution — Under $500")
axes[1].set_xlabel("Transaction Amount ($)")
axes[1].set_ylabel("Count")
axes[1].legend()

fig.tight_layout()
plt.show()

# Summary stats side by side
amount_stats = df.groupby(TARGET_COL)["Amount"].describe().T
amount_stats.columns = ["Legitimate", "Fraud"]
print(amount_stats.round(2).to_string())

**Insight:** Fraudulent transactions cluster at **lower amounts** (median ≈ $9.25 vs $22.00 for legitimate), suggesting attackers test with small transactions first. The model should treat amount as a meaningful signal, not just noise.

## 6 · Fraud Rate by Hour of Day

In [ ]:
# The 'Time' column = seconds elapsed since the first transaction.
# The dataset spans ~48 hours, so we derive hour-of-day (mod 24).
df["Hour"] = ((df["Time"] / 3600) % 24).astype(int)

# Compute fraud rate per hour
hourly = df.groupby("Hour").agg(
    total       = (TARGET_COL, "count"),
    fraud_count = (TARGET_COL, "sum")
)
hourly["fraud_rate_pct"] = (hourly["fraud_count"] / hourly["total"] * 100).round(4)

# --- Dual-axis plot ---
fig, ax1 = plt.subplots(figsize=(12, 5))

# Bar: total transactions per hour
ax1.bar(hourly.index, hourly["total"], color="#d5e8d4", edgecolor="gray",
        alpha=0.7, label="Total transactions")
ax1.set_xlabel("Hour of Day", fontsize=12)
ax1.set_ylabel("Total Transactions", fontsize=12, color="gray")
ax1.tick_params(axis="y", labelcolor="gray")

# Line: fraud rate
ax2 = ax1.twinx()
ax2.plot(hourly.index, hourly["fraud_rate_pct"], color="#e74c3c",
         marker="o", linewidth=2.5, label="Fraud rate (%)")
ax2.set_ylabel("Fraud Rate (%)", fontsize=12, color="#e74c3c")
ax2.tick_params(axis="y", labelcolor="#e74c3c")

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

ax1.set_title("Transaction Volume & Fraud Rate by Hour of Day", fontsize=14)
ax1.set_xticks(range(24))
fig.tight_layout()
plt.show()

# Print the table
print(hourly.to_string())

**Insight:** Fraud rate **spikes during late-night / early-morning hours** (roughly 1 AM – 4 AM) when transaction volume drops. Attackers exploit off-peak windows with fewer human reviewers — this temporal pattern is a strong feature for our model.

## 7 · Correlation Heatmap — Top Features vs Class

In [ ]:
# Compute correlations of all features with the target
all_features = PCA_FEATURES + ["Amount"]
corr_with_target = df[all_features + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)

# Select top 15 most correlated (by absolute value)
top_pos = corr_with_target.nlargest(8)
top_neg = corr_with_target.nsmallest(7)
top_features = pd.concat([top_neg, top_pos]).index.tolist()

print(f"Top {len(top_features)} features most correlated with fraud:\n")
for feat in top_features:
    val = corr_with_target[feat]
    direction = "🔴 +" if val > 0 else "🔵 "
    print(f"  {feat:>8s}  {direction}{val:.4f}")

In [ ]:
# Heatmap of correlations among top features + Class
cols_for_heatmap = top_features + [TARGET_COL]
corr_matrix = df[cols_for_heatmap].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            linewidths=0.5, square=True, ax=ax,
            cbar_kws={"shrink": 0.8, "label": "Pearson r"})
ax.set_title("Correlation Heatmap — Top Features vs Fraud (Class)", fontsize=14)
fig.tight_layout()
plt.show()

**Insight:** Features **V17, V14, V12, V10** show the strongest *negative* correlation with fraud (fraud drives these values lower), while **V4, V11** correlate *positively*. These PCA components will likely dominate the model's decision boundaries. The raw **Amount** feature has only a weak correlation, but becomes more useful after scaling.

## 8 · Cleanup & Next Steps

In [ ]:
# Drop the helper column we created
df.drop(columns=["Hour"], inplace=True)
print("✅ EDA complete — key takeaways:")
print("   • Extreme class imbalance (0.17 % fraud) → need SMOTE + AUPRC")
print("   • Fraud clusters at low amounts and off-peak hours")
print("   • V17, V14, V12, V10 are the strongest fraud signals")
print("   • No missing data — clean dataset ready for feature engineering")
print()
print("👉 Next notebook: 02_feature_engineering.ipynb")

---
### Key Takeaways for the Presentation

| Finding | Implication for Model |
|---|---|
| 0.17 % fraud rate (1:577 imbalance) | Use **SMOTE** oversampling; evaluate with **AUPRC**, not accuracy |
| Fraud amounts are smaller (median ~$9) | **Amount** is a useful feature after scaling |
| Fraud spikes at 1–4 AM | **Hour-of-day** should be an engineered feature |
| V17, V14, V12, V10 most correlated | These PCA components will drive the XGBoost model |
| Zero missing values | No imputation needed — straight to feature engineering |

---
*Fraud Shield on IBM Z — IBM Datathon 2026*